# Data Ingestions

In this notebook we will work with data ingestion of financial data and create a small datawarehouse. We will first ingest small data from growwapi and for now ingest data for One month i.e from `25th August 2025` till `25th August 2026`.
Since each sectors of the market has different amout of data we will split it into the top indexes.  They are:

1. Bank Nifty
2. Nifty IT
3. Nifty Phrama
4. Nifty Financial Services
5. Nifty FMCG
6. Nifty Infra
7. Nifty Oil and Gas
8. Nifty Media
9. Nifty Private bank
10. Nifty PSU bank
11. Nifty reality
12. Nifty Commodities
13. Nifty Health Care
14. Nifty  Consumer durables

The overall warehouse design will be similar to one but still I will keep adding the UML diagrams

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from growwapi import GrowwAPI
import os
from dotenv import load_dotenv
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

load = load_dotenv()

In [2]:
# Define explicit configuration properties

session = snowflake.connector.connect(
    account=os.getenv('account'),  # Do not include '.snowflakecomputing.com'
    user=os.getenv('user'),
    password=os.getenv('password'),
    warehouse=os.getenv('warehouse'),
    database=os.getenv('database'),
    schema=os.getenv('schema'),
    role=os.getenv('role')
)
print(session)

In [3]:
# Accessing API
GrowApi = os.getenv('GrowwAPi')
GrowScret = os.getenv('GrowwScret')

groww = GrowwAPI(GrowApi)

instrument_df = groww.get_all_instruments()


Ready to Groww!


## 1. Finance WH Design


In [4]:
success, nchunks, nrows, _ = write_pandas(
    conn=session,
    df=instrument_df,
    table_name="NiftyInstrument",
    auto_create_table=True,  # Generates the table layout
    overwrite=True           # Overwrites existing table
)


In [5]:
Equity_df = instrument_df[instrument_df['instrument_type'] == 'EQ']

success, nchunks, nrows, _ = write_pandas(
    conn=session,
    df=Equity_df,
    table_name="EquityInstruments",
    auto_create_table=True,  # Generates the table layout
    overwrite=True           # Overwrites existing table
)

/var/folders/zb/bl3vjblx59ggp4ygks32q_7r0000gn/T/ipykernel_53054/2064857547.py:3: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.core.indexes.base.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, nchunks, nrows, _ = write_pandas(


In [6]:
bank_nifty = [
    "HDFCBANK", "ICICIBANK", "SBIN", "AXISBANK", "KOTAKBANK", 
    "INDUSINDBK", "BANKBARODA", "PNB", "AUBANK", "FEDERALBNK", 
    "IDFCFIRSTB", "SBICARD"
]
BankNifty = instrument_df[instrument_df['trading_symbol'].isin(bank_nifty)]
BankNifty.head(10)

,exchange,exchange_token,trading_symbol,groww_symbol,name,instrument_type,segment,series,isin,underlying_symbol,...,expiry_date,strike_price,lot_size,tick_size,freeze_quantity,is_reserved,buy_allowed,sell_allowed,internal_trading_symbol,is_intraday
79350,NSE,21238,AUBANK,NSE-AUBANK,AU Small Fin. Bank,EQ,CASH,EQ,INE949L01017,NaN,...,NaN,NaN,1,0.1,NaN,NaN,1,1,AUBANK-EQ,1
79387,NSE,4668,BANKBARODA,NSE-BANKBARODA,Bank of Baroda,EQ,CASH,EQ,INE028A01039,NaN,...,NaN,NaN,1,0.01,NaN,NaN,1,1,BANKBARODA-EQ,1
79415,NSE,5900,AXISBANK,NSE-AXISBANK,Axis Bank,EQ,CASH,EQ,INE238A01034,NaN,...,NaN,NaN,1,0.1,NaN,NaN,1,1,AXISBANK-EQ,1
80007,NSE,1023,FEDERALBNK,NSE-FEDERALBNK,The Federal Bank,EQ,CASH,EQ,INE171A01029,NaN,...,NaN,NaN,1,0.05,NaN,NaN,1,1,FEDERALBNK-EQ,1
80317,NSE,1333,HDFCBANK,NSE-HDFCBANK,HDFC Bank,EQ,CASH,EQ,INE040A01034,NaN,...,NaN,NaN,1,0.05,NaN,NaN,1,1,HDFCBANK-EQ,1
80430,NSE,4963,ICICIBANK,NSE-ICICIBANK,ICICI Bank,EQ,CASH,EQ,INE090A01021,NaN,...,NaN,NaN,1,0.1,NaN,NaN,1,1,ICICIBANK-EQ,1
80474,NSE,11184,IDFCFIRSTB,NSE-IDFCFIRSTB,IDFC First Bank,EQ,CASH,EQ,INE092T01019,NaN,...,NaN,NaN,1,0.01,NaN,NaN,1,1,IDFCFIRSTB-EQ,1
80496,NSE,5258,INDUSINDBK,NSE-INDUSINDBK,IndusInd Bank,EQ,CASH,EQ,INE095A01012,NaN,...,NaN,NaN,1,0.1,NaN,NaN,1,1,INDUSINDBK-EQ,1
80758,NSE,1922,KOTAKBANK,NSE-KOTAKBANK,Kotak Mahindra Bank,EQ,CASH,EQ,INE237A01036,NaN,...,NaN,NaN,1,0.05,NaN,NaN,1,1,KOTAKBANK-EQ,1
81553,NSE,10666,PNB,NSE-PNB,PNB,EQ,CASH,EQ,INE160A01022,NaN,...,NaN,NaN,1,0.01,NaN,NaN,1,1,PNB-EQ,1


In [ ]:
bank_nifty = [
    "HDFCBANK", "ICICIBANK", "SBIN", "AXISBANK", "KOTAKBANK", 
    "INDUSINDBK", "BANKBARODA", "PNB", "AUBANK", "FEDERALBNK", 
    "IDFCFIRSTB", "SBICARD"
]

BankNifty = instrument_df[instrument_df['trading_symbol'].isin(bank_nifty)]
success, nchunks, nrows, _ = write_pandas(
    conn=session,
    df=BankNifty,
    table_name="Bank_Fact",
    auto_create_table=True,  # Generates the table layout
    overwrite=True           # Overwrites existing table
)

/var/folders/zb/bl3vjblx59ggp4ygks32q_7r0000gn/T/ipykernel_53054/679159584.py:1: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.core.indexes.base.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, nchunks, nrows, _ = write_pandas(


In [ ]:
nifty_financial_services = [
    "HDFCBANK", "ICICIBANK", "AXISBANK", "KOTAKBANK", "SBIN", 
    "BAJFINANCE", "BAJAJFINSV", "SBILIFE", "HDFCLIFE", "CHOLAFIN"
]

FinancialServicesDf = instrument_df[instrument_df['trading_symbol'].isin(nifty_financial_services)]
success, nchunks, nrows, _ = write_pandas(
    conn=session,
    df=FinancialServicesDf,
    table_name="Financial_Services_Fact",
    auto_create_table=True,  # Generates the table layout
    overwrite=True           # Overwrites existing table
)

/var/folders/zb/bl3vjblx59ggp4ygks32q_7r0000gn/T/ipykernel_53054/4020354521.py:6: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.core.indexes.base.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, nchunks, nrows, _ = write_pandas(


In [15]:
nifty_private_bank = [
    "HDFCBANK", "ICICIBANK", "AXISBANK", "KOTAKBANK", 
    "INDUSINDBK", "AUBANK", "FEDERALBNK", "IDFCFIRSTB", "RBLBANK", "CUB"
]

PrivateBankdf = instrument_df[instrument_df['trading_symbol'].isin(nifty_private_bank)]
success, nchunks, nrows, _ = write_pandas(
    conn=session,
    df=PrivateBankdf,
    table_name="Private_Bank_Fact",
    auto_create_table=True,  # Generates the table layout
    overwrite=True           # Overwrites existing table
)

/var/folders/zb/bl3vjblx59ggp4ygks32q_7r0000gn/T/ipykernel_53054/4154487461.py:7: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.core.indexes.base.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, nchunks, nrows, _ = write_pandas(


In [16]:
nifty_psu_bank = [
    "SBIN", "BANKBARODA", "PNB", "CANBK", "UNIONBANK", 
    "IOB", "BANKINDIA", "MAHABANK", "CENTRALBK", "UCOBANK"
]

PSUBank = instrument_df[instrument_df['trading_symbol'].isin(nifty_psu_bank)]
success, nchunks, nrows, _ = write_pandas(
    conn=session,
    df=PSUBank,
    table_name="PSU_Bank_Fact",
    auto_create_table=True,  # Generates the table layout
    overwrite=True           # Overwrites existing table
)

/var/folders/zb/bl3vjblx59ggp4ygks32q_7r0000gn/T/ipykernel_53054/1345295665.py:7: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.core.indexes.base.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, nchunks, nrows, _ = write_pandas(


In [18]:
session.close()